# Desafio Ciência e Governança de Dados

Desenvolvido por Estevão Augusto da Fonseca Santos, Graduando em Ciência de Computação

6° Período da Universidade Federal de Lavras

## Objetivos

"Como poderíamos avaliar e prever/visualizar os agentes/fenômenos que mais causam impactos socioeconômicos no Brasil?". Essa é a pergunta proposta pelo desafio. Para respondê-la, será preciso adquirir, organizar, explorar e visualizar os dados necessários para criar uma resposta à ele. O notebook "1_coleta_preparacao_dados.ipynb" está focado na coleta e preparação de dados.

In [13]:
import pandas as pd                 # Biblioteca para manipulação e análise de dados
import basedosdados as bd           # Biblioteca para acessar o datalake público do site BasedosDados
import os                           # Biblioteca para interação com arquivos e diretórios do sistema
from dotenv import load_dotenv      # Biblioteca para carregar variáveis de ambiente de arquivos .env

In [14]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))  # Adiciona raiz

# Define constantes que armazenam os caminhos que serão utilizados neste notebook
from config_path import RAW_DATA_DIRECTORY_PATH, PROCESSED_DATA_DIRECTORY_PATH, DATA_DIRECTORY_PATH

In [15]:
load_dotenv()  # Carrega os valores no arquivo .env

GOOGLE_CLOUD_ID_PROJECT = os.getenv("GOOGLE_CLOUD_ID_PROJECT") # Coloca o ID do projeto do Google Cloud numa constante

# Define constantes que armazenam os caminhos que serão utilizados neste notebook
# DATA_DIRECTORY_PATH = "../data"
# RAW_DATA_DIRECTORY_PATH = f"{DATA_DIRECTORY_PATH}/raw"
# PROCESSED_DATA_DIRECTORY_PATH = f"{DATA_DIRECTORY_PATH}/processed"

In [16]:
# Caso a pasta 'data' tenha sido excluida, o notebook cria ela
if not os.path.isdir(DATA_DIRECTORY_PATH):
    os.mkdir(DATA_DIRECTORY_PATH)
    os.mkdir(RAW_DATA_DIRECTORY_PATH)
    os.mkdir(PROCESSED_DATA_DIRECTORY_PATH)

## Obtençao de Dados e Preparação de Dados

#### Dados Geográficos do Brasil

##### Tamanho Geográfico UF do Brasil

In [4]:
# Aqui realiza-se a leitura de um arquivo de excel sobre dados territoriais do brasil, com o foco no território dos estados
df_uf = pd.read_excel(f"{RAW_DATA_DIRECTORY_PATH}/Dados_Tamanho_Brasil.ods", "AR_BR_UF_2024") 

df_uf.describe() # descreve o que é o df_uf
df_uf.info() # revela as colunas existentes e suas propriedades
df_uf.dropna(how='any', inplace=True)     # remove linhas totalmente vazias
df_uf.dropna(axis=1, how='all', inplace=True)  # remove colunas totalmente vazias
df_uf['NM_UF_SIGLA'] = df_uf['NM_UF_SIGLA'].astype(str)

df_uf.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_uf_2024.csv", index=False) # Converte o arquivo Excel para CSV para uso futuro

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CD_UF        28 non-null     object 
 1   NM_UF        27 non-null     object 
 2   NM_UF_SIGLA  27 non-null     object 
 3   AR_UF_2024   27 non-null     float64
dtypes: float64(1), object(3)
memory usage: 1.1+ KB


##### Tamanho Geográfico Municipial do Brasil 

In [ ]:
# Aqui realiza-se a leitura de um arquivo de excel sobre dados territoriais do brasil, com o foco no território dos municipios
df_mun = pd.read_excel(f"{RAW_DATA_DIRECTORY_PATH}/Dados_Tamanho_Brasil.ods", "AR_BR_MUN_2024")

df_mun.describe() # descreve o que é o df_mun
df_mun.info() # revela as colunas existentes e suas propriedades

df_mun.dropna(how='any', inplace=True)     # remove linhas totalmente vazias
df_mun.dropna(axis=1, how='all', inplace=True)  # remove colunas totalmente vazias
df_mun['NM_UF_SIGLA'] = df_mun['NM_UF_SIGLA'].astype(str)

df_mun.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_municipios_2024.csv", index=False) # Converte o arquivo Excel para CSV para uso futuro

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5575 entries, 0 to 5574
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CD_UF        5574 non-null   object 
 1   NM_UF        5573 non-null   object 
 2   NM_UF_SIGLA  5573 non-null   object 
 3   CD_MUN       5573 non-null   float64
 4   NM_MUN       5573 non-null   object 
 5   AR_MUN_2024  5573 non-null   float64
dtypes: float64(2), object(4)
memory usage: 261.5+ KB


##### Identificadores de Municipios do Brasil

In [ ]:
# Realiza-se a leitura de um arquivo CSV que contem dados que identificam Estados e Múnicipios de Brasil
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/traducao_municipios.csv")

# Seleciona-se as colunas desejadas para a integração de dados
df = df[df.columns.intersection(["id_municipio", "id_uf","sigla_uf","nome_uf","nome_regiao"])]

# Escreve o novo dataframe gerado num arquivo separado para consultas futuras
df.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/trad_municipio_tratados.csv", index=False)
df.head()

,id_municipio,id_uf,sigla_uf,nome_uf,nome_regiao
0,5101837,51,MT,Mato Grosso,Centro-Oeste
1,1100809,11,RO,Rondônia,Norte
2,1100338,11,RO,Rondônia,Norte
3,1100205,11,RO,Rondônia,Norte
4,1101104,11,RO,Rondônia,Norte


### População Brasileira

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do PIB por Municipio em 2021

query = """
  SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.sexo as sexo,
    dados.grupo_idade as grupo_idade,
    dados.populacao as populacao
FROM `basedosdados.br_ms_populacao.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano = 2021
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv")
df.head()

,ano,id_municipio,id_municipio_nome,sexo,grupo_idade,populacao
0,2021,1100015,Alta Floresta D'Oeste,feminino,0-4 anos,782
1,2021,1100015,Alta Floresta D'Oeste,masculino,0-4 anos,819
2,2021,1100015,Alta Floresta D'Oeste,feminino,10-14 anos,754
3,2021,1100015,Alta Floresta D'Oeste,masculino,10-14 anos,802
4,2021,1100015,Alta Floresta D'Oeste,feminino,15-19 anos,791


### Produto Interno Bruto (PIB) Por Municipio

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do PIB por Municipio em 2021

query = """
  SELECT
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.ano as ano,
    dados.pib as pib,
    dados.impostos_liquidos as impostos_liquidos,
    dados.va as va,
    dados.va_agropecuaria as va_agropecuaria,
    dados.va_industria as va_industria,
    dados.va_servicos as va_servicos,
    dados.va_adespss as va_adespss
FROM `basedosdados.br_ibge_pib.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano = 2021
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio.csv")
df.head()

,id_municipio,id_municipio_nome,ano,pib,impostos_liquidos,va,va_agropecuaria,va_industria,va_servicos,va_adespss
0,1100148,Nova Brasilândia D'Oeste,2021,548734000,38742000,509993000,228451000,24608000,114351000,142582000
1,1100940,Cujubim,2021,487480000,26537000,460943000,153670000,43842000,84313000,179118000
2,1101468,Pimenteiras do Oeste,2021,246209000,8203000,238005000,178073000,8909000,24664000,26359000
3,1500503,Almeirim,2021,688370000,66886000,621484000,88297000,217317000,118135000,197734000
4,1506203,Salinópolis,2021,641901000,74258000,567643000,21408000,58837000,301419000,185979000


### Produto Interno Bruto (PIB) Por UF

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do PIB por UF em 2020

query = """
  SELECT
    dados.ano as ano,
    dados.id_uf AS id_uf,
    diretorio_id_uf.sigla AS id_uf_sigla,
    diretorio_id_uf.nome AS id_uf_nome,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.pib as pib,
    dados.impostos_liquidos as impostos_liquidos,
    dados.va as va,
    dados.va_agropecuaria as va_agropecuaria,
    dados.va_industria as va_industria,
    dados.va_servicos as va_servicos,
    dados.va_adespss as va_adespss
FROM `basedosdados.br_ibge_pib.uf` AS dados
LEFT JOIN (SELECT DISTINCT id_uf,sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_id_uf
    ON dados.id_uf = diretorio_id_uf.id_uf
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
WHERE ano = 2020
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf.csv"):
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf.csv")
df.head()

,ano,id_uf,id_uf_sigla,id_uf_nome,sigla_uf,sigla_uf_nome,pib,impostos_liquidos,va,va_agropecuaria,va_industria,va_servicos,va_adespss
0,2020,11,RO,Rondônia,RO,Rondônia,51598741456,5360626779,46238114681,6891411669,8285675423,19060688172,12000339417
1,2020,12,AC,Acre,AC,Acre,16476370837,1679899793,14796471045,983531817,1191345379,6590543336,6031050521
2,2020,13,AM,Amazonas,AM,Amazonas,116019139386,20058422775,95960716614,5114449144,35839810630,34795836262,20210620577
3,2020,14,RR,Roraima,RR,Roraima,16024275696,1500036536,14524239158,1000907462,1706511471,5278754961,6538065265
4,2020,15,PA,Pará,PA,Pará,215935603795,18021964332,197913639459,19730656823,84173852308,56395092425,37614037902


In [ ]:
# Código abaixo consiste na integração dos datasets gerados até agora a fim de um arquivo CSV que
# contenha dados de todos os múnicipios, nisso inclui questões como identificadores (para integração de dados futura)
# valores economicos, demográficos e populacionais

df_populacao_brasileira = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv")
df_tamanho_municipios = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_municipios_2024.csv")
df_pip_por_municipio = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio.csv")

df_pip_por_municipio = df_pip_por_municipio[df_pip_por_municipio.columns.intersection(["id_municipio", 
                                                                                       "id_municipio_nome",
                                                "pib", 
                                                "impostos_liquidos",
                                                "va",
                                                "va_agropecuaria",
                                                "va_industria",
                                                "va_servicos",
                                                "va_adespss"])]


# Valores foram normalizados para formato de Milhoes (1.000.000)
df_pip_por_municipio["pib"] = df_pip_por_municipio["pib"] / 1_000_000
df_pip_por_municipio["impostos_liquidos"] = df_pip_por_municipio["va"] / 1_000_000
df_pip_por_municipio["va"] = df_pip_por_municipio["va"] / 1_000_000
df_pip_por_municipio["va_agropecuaria"] = df_pip_por_municipio["va_agropecuaria"] / 1_000_000
df_pip_por_municipio["va_industria"] = df_pip_por_municipio["va_industria"] / 1_000_000
df_pip_por_municipio["va_servicos"] = df_pip_por_municipio["va_servicos"] / 1_000_000
df_pip_por_municipio["va_adespss"] = df_pip_por_municipio["va_adespss"] / 1_000_000

df_tamanho_municipios = df_tamanho_municipios[df_tamanho_municipios.columns.intersection(["CD_MUN", "AR_MUN_2024", "NM_UF_SIGLA", "NM_UF"])]

pop_municipio = df_populacao_brasileira.groupby(['id_municipio'])['populacao'].sum().reset_index()
pop_municipio.rename(columns={'populacao': 'populacao_total'}, inplace=True)
pop_municipio = pop_municipio.merge(df_pip_por_municipio, on='id_municipio', how='inner')

pop_municipio = pd.merge(pop_municipio, df_tamanho_municipios, left_on="id_municipio", right_on="CD_MUN", how="inner")
pop_municipio.drop("CD_MUN", axis=1, inplace=True)

pop_estado = pop_municipio.groupby("NM_UF_SIGLA", as_index=False)["populacao_total"].sum()

# Arquivo CSV abaixo contem diversos dados de múnicipios
pop_municipio.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_mun.csv")

populacao_por_estado = pop_municipio.groupby('NM_UF_SIGLA', as_index=False)['populacao_total'].sum()
populacao_por_estado.rename(columns={'populacao_total': 'populacao_estado'}, inplace=True)

del df_pip_por_municipio

In [ ]:
# Código abaixo consiste na integração dos datasets gerados até agora a fim de um arquivo CSV que
# contenha dados de todos os UFs, nisso inclui questões como identificadores (para integração de dados futura)
# valores economicos, demográficos e populacionais

df_tamanho_uf = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_uf_2024.csv")
df_tamanho_uf = df_tamanho_uf[["NM_UF_SIGLA", "AR_UF_2024"]]
df_pip_por_uf = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf.csv")

df_pip_por_uf = df_pip_por_uf[df_pip_por_uf.columns.intersection(["id_uf_sigla", 
                                                "pib", 
                                                "impostos_liquidos",
                                                "va",
                                                "va_agropecuaria",
                                                "va_industria",
                                                "va_servicos",
                                                "va_adespss"])]


# Valores foram normalizados para formato de Milhoes (1.000.000)
df_pip_por_uf["pib"] = df_pip_por_uf["pib"] / 1_000_000
df_pip_por_uf["impostos_liquidos"] = df_pip_por_uf["va"] / 1_000_000
df_pip_por_uf["va"] = df_pip_por_uf["va"] / 1_000_000
df_pip_por_uf["va_agropecuaria"] = df_pip_por_uf["va_agropecuaria"] / 1_000_000
df_pip_por_uf["va_industria"] = df_pip_por_uf["va_industria"] / 1_000_000
df_pip_por_uf["va_servicos"] = df_pip_por_uf["va_servicos"] / 1_000_000
df_pip_por_uf["va_adespss"] = df_pip_por_uf["va_adespss"] / 1_000_000

df_pip_por_uf = pd.merge(df_pip_por_uf, df_tamanho_uf, left_on="id_uf_sigla", right_on="NM_UF_SIGLA", how="left")
df_pip_por_uf = pd.merge(df_pip_por_uf, populacao_por_estado, left_on="id_uf_sigla", right_on="NM_UF_SIGLA", how="inner")

# Arquivo CSV abaixo contem diversos dados de UFs
df_pip_por_uf.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_uf.csv")

In [12]:
del df_pip_por_uf
del df_tamanho_uf

### Indice de Gini

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do Indice de Gini a nível Estadual
# Dataset foi criado em 2022

query = """
  SELECT
    dados.id_uf as id_uf,
    dados.ano as ano,
    dados.gini_pib as gini_pib,
    dados.gini_va_agro as gini_va_agro,
    dados.gini_va_industria as gini_va_industria,
    dados.gini_va_servicos as gini_va_servicos,
    dados.gini_va_adespss as gini_va_adespss
FROM `basedosdados.br_ibge_pib.gini` AS dados
WHERE ano >= 2018 AND ano <= 2022
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/indice_de_gini_uf.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/indice_de_gini_uf.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/indice_de_gini_uf.csv")
df.head()

### Censo 2022 - Alfabetização por Sexo, Raça e Grupo de Idade

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV da Alfabetização de Brasileiros dividida em grupos por Sexo, Raça e Grupo de Idade. 
# Dataset foi criado em 2022

query = """
  SELECT
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.cor_raca as cor_raca,
    dados.sexo as sexo,
    dados.grupo_idade as grupo_idade,
    dados.alfabetizacao as alfabetizacao,
    dados.populacao as populacao
FROM `basedosdados.br_ibge_censo_2022.alfabetizacao_grupo_idade_sexo_raca` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizacao_por_sexo,raca_e_idade.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizacao_por_sexo,raca_e_idade.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizacao_por_sexo,raca_e_idade.csv")
df.head()

,id_municipio,id_municipio_nome,cor_raca,sexo,grupo_idade,alfabetizacao,populacao
0,4123501,Santa Helena,Parda,Mulheres,15 a 19 anos,Alfabetizadas,256
1,5205513,Cocalzinho de Goiás,Branca,Mulheres,15 a 19 anos,Alfabetizadas,256
2,1502707,Conceição do Araguaia,Preta,Homens,15 a 19 anos,Alfabetizadas,256
3,2703601,Japaratinga,Parda,Homens,15 a 19 anos,Alfabetizadas,256
4,2707305,Porto Calvo,Branca,Mulheres,15 a 19 anos,Alfabetizadas,256


In [ ]:
# Ler o CSV
df_alfabeticao_csv = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizacao_por_sexo,raca_e_idade.csv")

# Filtrar apenas pessoas alfabetizadas
df_alfabetizados = df_alfabeticao_csv[df_alfabeticao_csv['alfabetizacao'] == 'Alfabetizadas']

# Agrupar por município e somar a população
tabela_total_alfabetizados = df_alfabetizados.groupby(
    ['id_municipio', 'id_municipio_nome'], as_index=False
)['populacao'].sum()

# Renomear coluna para ficar mais claro
tabela_total_alfabetizados.rename(columns={'populacao': 'total_alfabetizados'}, inplace=True)

# Salvar em CSV
tabela_total_alfabetizados.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/alfabetizacao_processada.csv", index=False)


In [15]:
del df_alfabeticao_csv

### Sinopses Estatísticas da Educação Básica - Sexo Raça Cor

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV da Educação Básica, a qual conta com o total de matrículas por município para todas as etapas de ensino, sexo e raça/cor
# Dataset foi criado em 2024

query = """
  SELECT
    dados.ano as ano,
    dados.sigla_uf as sigla_uf,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.etapa_ensino as etapa_ensino,
    dados.sexo as sexo,
    dados.raca_cor as raca_cor,
    dados.quantidade_matricula as quantidade_matricula
FROM `basedosdados.br_inep_sinopse_estatistica_educacao_basica.sexo_raca_cor` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano = 2021
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{PROCESSED_DATA_DIRECTORY_PATH}/educacao_basica_sexo_raca_cor.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/educacao_basica_sexo_raca_cor.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/educacao_basica_sexo_raca_cor.csv")
df.head()

,ano,sigla_uf,id_municipio,id_municipio_nome,etapa_ensino,sexo,raca_cor,quantidade_matricula
0,2021,AC,1200013,Acrelândia,Educação Infantil – Pré-Escola,Masculino,Amarela,1
1,2021,AC,1200013,Acrelândia,Ensino Fundamental – Anos Iniciais,Masculino,Amarela,2
2,2021,AC,1200013,Acrelândia,Ensino Médio Regular,Masculino,Amarela,0
3,2021,AC,1200013,Acrelândia,Educação Infantil – Creche,Feminino,Amarela,0
4,2021,AC,1200013,Acrelândia,Educação Especial – Classes Exclusivas,Masculino,Amarela,0


In [ ]:
# calcula a quantidade de matriculas por ensino dividio pelo sexo
matriculas_por_ensino_sexo = df.groupby(
    ['etapa_ensino', 'sexo'], as_index=False
)['quantidade_matricula'].sum()

# Renomear a coluna para algo mais claro
matriculas_por_ensino_sexo.rename(columns={'quantidade_matricula': 'total_matriculas'}, inplace=True)

print(matriculas_por_ensino_sexo)

# Escrever no arquivo csv
matriculas_por_ensino_sexo.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/quantidade_total_matriculas_alfabetizacao.csv", 
                                  index=False)
del matriculas_por_ensino_sexo

                              etapa_ensino       sexo  total_matriculas
0       Educação Especial – Classes Comuns   Feminino            394234
1       Educação Especial – Classes Comuns  Masculino            800610
2   Educação Especial – Classes Exclusivas   Feminino             61793
3   Educação Especial – Classes Exclusivas  Masculino             94284
4               Educação Infantil – Creche   Feminino           1652149
5               Educação Infantil – Creche  Masculino           1765061
6           Educação Infantil – Pré-Escola   Feminino           2400819
7           Educação Infantil – Pré-Escola  Masculino           2501370
8                    Educação Profissional   Feminino           1092961
9                    Educação Profissional  Masculino            799497
10      Educação de Jovens e Adultos (EJA)   Feminino           1547580
11      Educação de Jovens e Adultos (EJA)  Masculino           1414742
12        Ensino Fundamental – Anos Finais   Feminino           

In [ ]:
# Calcula a quantidade de matriculas baseado em grupos de certos criterios (municipio, etapa de ensino, e sexo)
matriculas_por_ensino_sexo = df.groupby(
    ['id_municipio', 'id_municipio_nome', 'etapa_ensino', 'sexo'], as_index=False
)['quantidade_matricula'].sum()

# Renomear a coluna para algo mais claro
matriculas_por_ensino_sexo.rename(columns={'quantidade_matricula': 'total_matriculas'}, inplace=True)

print(matriculas_por_ensino_sexo)

# Escreve o dataframe num arquivo CSV
matriculas_por_ensino_sexo.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/quantidade_total_matriculas_alfabetizacao_dividida_mun.csv", 
                                  index=False)
del matriculas_por_ensino_sexo

        id_municipio      id_municipio_nome  \
0            1100015  Alta Floresta D'Oeste   
1            1100015  Alta Floresta D'Oeste   
2            1100015  Alta Floresta D'Oeste   
3            1100015  Alta Floresta D'Oeste   
4            1100015  Alta Floresta D'Oeste   
...              ...                    ...   
100255       5300108               Brasília   
100256       5300108               Brasília   
100257       5300108               Brasília   
100258       5300108               Brasília   
100259       5300108               Brasília   

                                  etapa_ensino       sexo  total_matriculas  
0           Educação Especial – Classes Comuns   Feminino                47  
1           Educação Especial – Classes Comuns  Masculino                80  
2       Educação Especial – Classes Exclusivas   Feminino                 3  
3       Educação Especial – Classes Exclusivas  Masculino                 3  
4                   Educação Infantil – Crech

## Geração de Tabelas do Brasil, Minas Gerais e Lavras

### Tabela de Informações Gerais do Brasil

In [ ]:
df_populacao = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv")
df_pib_uf = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf.csv")
df_pib_mun = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio.csv")
df_tamanho_uf = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_uf.csv")
df_tamanho_mun = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_mun.csv")

In [ ]:
# Muito dos valores da tabela são representados por milhoes, a fim de tornar sua visualização mais facil
populacao_total_em_milhoes = ((df_populacao['populacao']).sum())
tamanho_brasil_km2 = (df_tamanho_uf['AR_UF_2024'].sum())
pib_total_brasil = (df_pib_uf['pib'].sum() )
valor_agro = (df_pib_uf['va_agropecuaria'].sum())
valor_industria = (df_pib_uf['va_industria'].sum() )
valor_servicos = (df_pib_uf['va_servicos'].sum())
valor_adespss = (df_pib_uf['va_adespss'].sum())
pib_per_capta_brasil = (pib_total_brasil / populacao_total_em_milhoes)
densidade_populacional = (populacao_total_em_milhoes / tamanho_brasil_km2)
valor_total = valor_agro + valor_industria + valor_servicos + valor_adespss

brasil_info = pd.DataFrame({
    "populacao_total" : [populacao_total_em_milhoes],
    "densidade_populacional" : [densidade_populacional],
    "tamanho_brasil_km2" : [tamanho_brasil_km2],
    "pib_total_brasil" : [pib_total_brasil],
    "pib_per_capta" : [pib_per_capta_brasil],
    "valor_agro" : [valor_agro],
    "valor_industria" : [valor_industria],
    "valor_servicos" : [valor_servicos],
    "valor_adespss" : [valor_adespss],
    "valor_total" : [valor_total]
})

# Escreve o dataframe num arquivo CSV
brasil_info.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/brasil_info.csv", index=False)

### Tabela de Informações de Minas Gerais

In [ ]:
estado_minas_gerais = (df_tamanho_uf.loc[df_tamanho_uf["id_uf_sigla"] == "MG"])

tamanho_mg_km2 = estado_minas_gerais["AR_UF_2024"]
pib_total_mg = estado_minas_gerais["pib"]
valor_agro = estado_minas_gerais["va_agropecuaria"]
valor_industria = estado_minas_gerais["va_industria"]
valor_servicos = estado_minas_gerais["va_servicos"]
valor_adespss = estado_minas_gerais["va_adespss"]
pib_per_capta_mg = (pib_total_mg / estado_minas_gerais["populacao_estado"])
densidade_populacional = (estado_minas_gerais["populacao_estado"] / tamanho_mg_km2)
valor_total = valor_agro + valor_industria + valor_servicos + valor_adespss

mg_info = pd.DataFrame({
    "tamanho_km2" : [tamanho_mg_km2],
    "populacao": [estado_minas_gerais["populacao_estado"]],
    "pib_per_capita_mg": [pib_per_capta_mg],
    "pib_mg" : [pib_total_mg] ,
    "setor_agro": [valor_agro],
    "setor_industria": [valor_industria],
    "setor_servicos": [valor_servicos],
    "setor_adespss": [valor_adespss],
    "area_km2": [estado_minas_gerais["AR_UF_2024"]],
    "densidade_populacional": [densidade_populacional]
})

# Escreve o dataframe num arquivo CSV
mg_info.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/mg_info.csv", index=False)

### Tabela de Informações de Lavras

In [ ]:
mun_lavras = (df_tamanho_mun.loc[df_tamanho_mun["id_municipio"] == 3138203])

tamanho_mg_km2 = mun_lavras["AR_MUN_2024"]
pib_total_mg = mun_lavras["pib"]
valor_agro = mun_lavras["va_agropecuaria"]
valor_industria = mun_lavras["va_industria"]
valor_servicos = mun_lavras["va_servicos"]
valor_adespss = mun_lavras["va_adespss"]
pib_per_capta_mg = (pib_total_mg / mun_lavras["populacao_total"])
densidade_populacional = (mun_lavras["populacao_total"] / tamanho_mg_km2)
valor_total = valor_agro + valor_industria + valor_servicos + valor_adespss

lavras_info = pd.DataFrame({
    "tamanho_km2" : [tamanho_mg_km2],
    "populacao": [mun_lavras["populacao_total"]],
    "pib_per_capita_mg": [pib_per_capta_mg],
    "pib_mg" : [pib_total_mg] ,
    "setor_agro": [valor_agro],
    "setor_industria": [valor_industria],
    "setor_servicos": [valor_servicos],
    "setor_adespss": [valor_adespss],
    "area_km2": [mun_lavras["AR_MUN_2024"]],
    "densidade_populacional": [densidade_populacional]
})

# Escreve o dataframe num arquivo CSV
lavras_info.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/lavras_info.csv", index=False)